# Silver -> Gold (Star Schema / Kimball)

Notebook que materializa o **modelo estrela** na camada Gold a partir da Silver.
A logica vive em `src/04_modelagem_gold/silver_to_gold.py`; aqui apenas a importamos,
executamos e inspecionamos o resultado.

Pre-requisito: a Silver ja precisa ter sido populada por `bronze_to_silver.py`.

In [ ]:
import sys
from pathlib import Path

# Dentro do container Jupyter o projeto fica em /home/jovyan/work.
SRC = Path("/home/jovyan/work/src")
if not SRC.exists():
    SRC = Path.cwd().parent / "src"  # fallback para execucao local
sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv()

import importlib
gold = importlib.import_module("04_modelagem_gold.silver_to_gold")
from utils.spark_config import build_spark_session

In [ ]:
spark = build_spark_session(app_name="silver-to-gold-notebook")
config = gold.GoldConfig.from_env()
config

## Constroi todas as dimensoes e fatos

Constroi as 5 dimensoes e os 4 fatos e grava cada um como Delta em `s3a://gold/<obj>/`.

In [ ]:
results = gold.run(spark, config, list(gold.ALL_OBJECTS))
for r in results:
    print(r)

## Inspeciona o resultado

Le os Delta da Gold e registra views temporarias para consultar com SQL
(as mesmas de `src/04_modelagem_gold/consultas_analiticas.sql`).

In [ ]:
for obj in gold.ALL_OBJECTS:
    path = f"s3a://{config.gold_bucket}/{obj}"
    spark.read.format("delta").load(path).createOrReplaceTempView(obj)

spark.table("dim_streamer").show(5, truncate=False)
spark.table("dim_tempo").orderBy("sk_tempo").show(5, truncate=False)

In [ ]:
# Top 10 streamers por doacoes (consulta 1 do consultas_analiticas.sql).
spark.sql("""
    SELECT s.nome_streamer, ROUND(SUM(f.valor), 2) AS total_doacoes, COUNT(*) AS qtd
    FROM fato_doacoes f
    JOIN dim_streamer s ON s.sk_streamer = f.sk_streamer
    GROUP BY s.nome_streamer
    ORDER BY total_doacoes DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
spark.stop()